# Interactive HMM Parameter Optimization

This notebook provides interactive hyperparameter tuning for Hidden Markov Models using ipywidgets. It enables real-time parameter adjustment and model comparison for optimal regime detection.

## Objectives
- Interactive parameter tuning with real-time feedback
- Hyperparameter optimization using grid search and Bayesian methods
- Model comparison and selection based on multiple criteria
- Export optimized model configurations for production use

## Setup and Imports

In [40]:
# Setup notebook environment
import sys
from pathlib import Path

# Add utils to path
sys.path.append('utils')

from utils.notebook_utils import setup_notebook_environment, check_dependencies
from utils.data_loaders import load_sample_data, preprocess_signals, create_multivariate_observations
from utils.plotting_helpers import plot_model_comparison, plot_transition_matrix, plot_state_probabilities

# Setup environment
project_root = setup_notebook_environment()
check_dependencies()

✓ Notebook environment configured
✓ Project root: /home/nipung2010/fictional-enigma
🔍 Checking dependencies...
✓ All required dependencies available


True

In [41]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# HMM and optimization libraries
from hmmlearn import hmm as hmmlearn_hmm
from sklearn.model_selection import ParameterGrid, cross_val_score
from sklearn.metrics import silhouette_score

# Try to import optimization libraries
try:
    from skopt import gp_minimize
    from skopt.space import Integer, Categorical
    from skopt.utils import use_named_args
    SKOPT_AVAILABLE = True
    print("✅ scikit-optimize available for Bayesian optimization")
except ImportError:
    SKOPT_AVAILABLE = False
    print("⚠️  scikit-optimize not available - will use grid search only")

# Display settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

✅ scikit-optimize available for Bayesian optimization


## Load Data

In [42]:
# Load processed data
try:
    # Try to load processed data from previous notebooks
    df_signals = pd.read_parquet('processed_data/signals_processed.parquet')
    observations = np.load('processed_data/observations.npy')
    print("✅ Loaded processed data from previous notebooks")
    print(f"   Signals shape: {df_signals.shape}")
    print(f"   Observations shape: {observations.shape}")
except FileNotFoundError:
    print("📊 Creating sample data for parameter optimization...")
    df_signals = load_sample_data(n_samples=1500, n_features=3, random_state=42)
    df_processed = preprocess_signals(df_signals, method='standardize')
    observations = create_multivariate_observations(df_processed)
    df_signals = df_processed  # Use processed version
    print(f"   Generated observations shape: {observations.shape}")

# Split data for optimization
split_ratio = 0.8
split_idx = int(len(observations) * split_ratio)

train_data = observations[:split_idx]
val_data = observations[split_idx:]

print(f"\n📊 Data Split for Optimization:")
print(f"   Training: {train_data.shape}")
print(f"   Validation: {val_data.shape}")
print(f"   Features: {list(df_signals.columns)}")

✅ Loaded processed data from previous notebooks
   Signals shape: (2000, 3)
   Observations shape: (2000, 3)

📊 Data Split for Optimization:
   Training: (1600, 3)
   Validation: (400, 3)
   Features: ['s_signal_1', 's_signal_2', 's_signal_3']


## Interactive Parameter Tuning Interface

In [43]:
# Create interactive parameter tuning class
class HMMParameterTuner:
    """Interactive HMM parameter tuning interface."""
    
    def __init__(self, train_data, val_data, df_signals):
        self.train_data = train_data
        self.val_data = val_data
        self.df_signals = df_signals
        self.results = {}
        self.current_model = None
        self.current_config = None
        
    def create_tuning_interface(self):
        """Create interactive parameter tuning interface."""
        
        # Parameter widgets
        self.n_states_slider = widgets.IntSlider(
            value=3, min=2, max=8, step=1,
            description='States:', 
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        
        self.covariance_dropdown = widgets.Dropdown(
            options=['full', 'diag', 'spherical'],
            value='full',
            description='Covariance:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        
        self.iterations_slider = widgets.IntSlider(
            value=100, min=50, max=500, step=25,
            description='Max Iterations:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        
        self.tolerance_slider = widgets.FloatLogSlider(
            value=1e-2, base=10, min=-4, max=-1, step=0.1,
            description='Tolerance:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        
        self.random_seed_slider = widgets.IntSlider(
            value=42, min=1, max=100, step=1,
            description='Random Seed:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        
        # Control buttons
        self.train_button = widgets.Button(
            description='🚀 Train Model',
            button_style='success',
            layout=widgets.Layout(width='150px', height='40px')
        )
        
        self.compare_button = widgets.Button(
            description='📊 Compare All',
            button_style='info',
            layout=widgets.Layout(width='150px', height='40px')
        )
        
        self.reset_button = widgets.Button(
            description='🔄 Reset',
            button_style='warning',
            layout=widgets.Layout(width='150px', height='40px')
        )
        
        # Output areas
        self.output_area = widgets.Output(layout=widgets.Layout(height='400px'))
        self.plot_area = widgets.Output(layout=widgets.Layout(height='500px'))
        
        # Progress bar
        self.progress_bar = widgets.IntProgress(
            value=0, min=0, max=100,
            description='Progress:',
            bar_style='info',
            layout=widgets.Layout(width='400px')
        )
        
        # Bind events
        self.train_button.on_click(self._on_train_clicked)
        self.compare_button.on_click(self._on_compare_clicked)
        self.reset_button.on_click(self._on_reset_clicked)
        
        # Layout
        parameter_box = widgets.VBox([
            widgets.HTML("<h3>🔧 HMM Parameters</h3>"),
            self.n_states_slider,
            self.covariance_dropdown,
            self.iterations_slider,
            self.tolerance_slider,
            self.random_seed_slider,
            widgets.HTML("<br>"),
            widgets.HBox([self.train_button, self.compare_button, self.reset_button]),
            self.progress_bar
        ])
        
        results_box = widgets.VBox([
            widgets.HTML("<h3>📊 Results</h3>"),
            self.output_area
        ])
        
        plots_box = widgets.VBox([
            widgets.HTML("<h3>📈 Visualizations</h3>"),
            self.plot_area
        ])
        
        main_interface = widgets.HBox([
            parameter_box,
            widgets.VBox([results_box, plots_box])
        ])
        
        return main_interface
    def _on_train_clicked(self, button):
        """Handle train button click."""
        with self.output_area:
            clear_output(wait=True)
            
            # Get current parameters
            config = {
                'n_components': self.n_states_slider.value,
                'covariance_type': self.covariance_dropdown.value,
                'n_iter': self.iterations_slider.value,
                'tol': self.tolerance_slider.value,
                'random_state': self.random_seed_slider.value
            }
            
            print(f"🚀 Training HMM with configuration:")
            for key, value in config.items():
                print(f"   {key}: {value}")
            
            # Update progress
            self.progress_bar.value = 20
            
            try:
                # Train model
                start_time = time.time()
                model = hmmlearn_hmm.GaussianHMM(**config)
                model.fit(self.train_data)
                training_time = time.time() - start_time
                
                self.progress_bar.value = 60
                
                # Evaluate model
                train_ll = model.score(self.train_data)
                val_ll = model.score(self.val_data)
                
                # Calculate metrics
                n_params = self._count_parameters(model, config)
                aic = 2 * n_params - 2 * val_ll
                bic = np.log(len(self.val_data)) * n_params - 2 * val_ll
                
                self.progress_bar.value = 80
                
                # Store results
                config_key = f"{config['n_components']}_{config['covariance_type']}_{config['n_iter']}"
                results = {
                    'config': config,
                    'model': model,
                    'train_ll': train_ll,
                    'val_ll': val_ll,
                    'aic': aic,
                    'bic': bic,
                    'training_time': training_time,
                    'converged': model.monitor_.converged,
                    'iterations': model.monitor_.iter,
                    'n_parameters': n_params
                }
                
                self.results[config_key] = results
                self.current_model = model
                self.current_config = config
                
                self.progress_bar.value = 100
                
                # Display results
                print(f"\n✅ Training completed in {training_time:.2f} seconds!")
                print(f"   Converged: {model.monitor_.converged} (iterations: {model.monitor_.iter})")
                print(f"   Training log-likelihood: {train_ll:.4f}")
                print(f"   Validation log-likelihood: {val_ll:.4f}")
                print(f"   AIC: {aic:.4f}")
                print(f"   BIC: {bic:.4f}")
                print(f"   Parameters: {n_params}")
                
                # Generate visualizations
                self._update_plots(model, config)
                
            except Exception as e:
                print(f"❌ Training failed: {str(e)}")
                self.progress_bar.value = 0
    
    def _on_compare_clicked(self, button):
        """Handle compare button click."""
        with self.output_area:
            clear_output(wait=True)
            
            if len(self.results) < 2:
                print("⚠️  Need at least 2 trained models for comparison")
                print("   Train more models with different parameters first")
                return
            
            print(f"📊 Comparing {len(self.results)} trained models...\n")
            
            # Create comparison DataFrame
            comparison_data = []
            for key, result in self.results.items():
                comparison_data.append({
                    'Configuration': key,
                    'States': result['config']['n_components'],
                    'Covariance': result['config']['covariance_type'],
                    'Val Log-Likelihood': result['val_ll'],
                    'AIC': result['aic'],
                    'BIC': result['bic'],
                    'Training Time': result['training_time'],
                    'Converged': result['converged']
                })
            
            df_comparison = pd.DataFrame(comparison_data)
            
            # Sort by BIC (lower is better)
            df_comparison = df_comparison.sort_values('BIC')
            
            print("🏆 Model Comparison Results (sorted by BIC):")
            print("=" * 80)
            display(df_comparison.round(4))
            
            # Best model
            best_config = df_comparison.iloc[0]['Configuration']
            best_bic = df_comparison.iloc[0]['BIC']
            
            print(f"\n🥇 Best Model: {best_config}")
            print(f"   BIC: {best_bic:.4f}")
            
            # Generate comparison plots
            self._plot_comparison(df_comparison)
    
    def _on_reset_clicked(self, button):
        """Handle reset button click."""
        self.results = {}
        self.current_model = None
        self.current_config = None
        self.progress_bar.value = 0
        
        with self.output_area:
            clear_output(wait=True)
            print("🔄 Results cleared. Ready for new experiments.")
        
        with self.plot_area:
            clear_output(wait=True)
    def _update_plots(self, model, config):
        """Update visualization plots."""
        with self.plot_area:
            clear_output(wait=True)
            
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            
            # 1. Transition matrix
            ax1 = axes[0, 0]
            im1 = ax1.imshow(model.transmat_, cmap='Blues', aspect='auto')
            ax1.set_title('Transition Matrix')
            ax1.set_xlabel('To State')
            ax1.set_ylabel('From State')
            
            # Add text annotations
            for i in range(config['n_components']):
                for j in range(config['n_components']):
                    text = ax1.text(j, i, f'{model.transmat_[i, j]:.3f}',
                                   ha="center", va="center", 
                                   color="white" if model.transmat_[i, j] > 0.5 else "black")
            
            plt.colorbar(im1, ax=ax1, shrink=0.8)
            
            # 2. State probabilities (sample)
            ax2 = axes[0, 1]
            sample_size = min(200, len(self.val_data))
            state_probs = model.predict_proba(self.val_data[:sample_size])
            
            for i in range(config['n_components']):
                ax2.plot(state_probs[:, i], label=f'State {i}', alpha=0.8)
            
            ax2.set_title(f'State Probabilities (First {sample_size} validation points)')
            ax2.set_xlabel('Time')
            ax2.set_ylabel('Probability')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
            
            # 3. Model means
            ax3 = axes[1, 0]
            means = model.means_
            x_pos = np.arange(means.shape[1])
            width = 0.8 / config['n_components']
            
            for i in range(config['n_components']):
                ax3.bar(x_pos + i*width, means[i], width, label=f'State {i}', alpha=0.8)
            
            ax3.set_title('State Means')
            ax3.set_xlabel('Feature')
            ax3.set_ylabel('Mean Value')
            ax3.set_xticks(x_pos + width * (config['n_components']-1) / 2)
            ax3.set_xticklabels([f'F{i}' for i in range(means.shape[1])])
            ax3.legend()
            ax3.grid(True, alpha=0.3)
            
            # 4. Training convergence (if available)
            ax4 = axes[1, 1]
            if hasattr(model.monitor_, 'history'):
                ax4.plot(model.monitor_.history)
                ax4.set_title('Training Convergence')
                ax4.set_xlabel('Iteration')
                ax4.set_ylabel('Log-likelihood')
                ax4.grid(True, alpha=0.3)
            else:
                ax4.text(0.5, 0.5, 'Convergence history\nnot available', 
                         ha='center', va='center', transform=ax4.transAxes)
                ax4.set_title('Training Convergence')
            
            plt.tight_layout()
            plt.show()
    
    def _plot_comparison(self, df_comparison):
        """Plot model comparison charts."""
        with self.plot_area:
            clear_output(wait=True)
            
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            
            # 1. BIC comparison
            ax1 = axes[0, 0]
            bars1 = ax1.bar(range(len(df_comparison)), df_comparison['BIC'], alpha=0.8)
            ax1.set_title('BIC Comparison (Lower is Better)')
            ax1.set_xlabel('Model')
            ax1.set_ylabel('BIC')
            ax1.set_xticks(range(len(df_comparison)))
            ax1.set_xticklabels(df_comparison['Configuration'], rotation=45, ha='right')
            ax1.grid(True, alpha=0.3)
            
            # 2. Log-likelihood comparison
            ax2 = axes[0, 1]
            bars2 = ax2.bar(range(len(df_comparison)), df_comparison['Val Log-Likelihood'], alpha=0.8, color='orange')
            ax2.set_title('Validation Log-Likelihood (Higher is Better)')
            ax2.set_xlabel('Model')
            ax2.set_ylabel('Log-Likelihood')
            ax2.set_xticks(range(len(df_comparison)))
            ax2.set_xticklabels(df_comparison['Configuration'], rotation=45, ha='right')
            ax2.grid(True, alpha=0.3)
            
            # 3. Training time comparison
            ax3 = axes[1, 0]
            bars3 = ax3.bar(range(len(df_comparison)), df_comparison['Training Time'], alpha=0.8, color='green')
            ax3.set_title('Training Time Comparison')
            ax3.set_xlabel('Model')
            ax3.set_ylabel('Time (seconds)')
            ax3.set_xticks(range(len(df_comparison)))
            ax3.set_xticklabels(df_comparison['Configuration'], rotation=45, ha='right')
            ax3.grid(True, alpha=0.3)
            
            # 4. States vs Performance scatter
            ax4 = axes[1, 1]
            colors = ['red' if not conv else 'blue' for conv in df_comparison['Converged']]
            scatter = ax4.scatter(df_comparison['States'], df_comparison['BIC'], 
                                c=colors, alpha=0.7, s=100)
            ax4.set_title('States vs BIC (Blue=Converged, Red=Not Converged)')
            ax4.set_xlabel('Number of States')
            ax4.set_ylabel('BIC')
            ax4.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
    
    def _count_parameters(self, model, config):
        """Count model parameters for AIC/BIC calculation."""
        n_states = config['n_components']
        n_features = model.means_.shape[1]
        
        # Transition matrix parameters
        n_trans_params = n_states * (n_states - 1)
        
        # Start probability parameters
        n_start_params = n_states - 1
        
        # Emission parameters (means + covariances)
        n_mean_params = n_states * n_features
        
        if config['covariance_type'] == 'full':
            n_cov_params = n_states * n_features * (n_features + 1) // 2
        elif config['covariance_type'] == 'diag':
            n_cov_params = n_states * n_features
        elif config['covariance_type'] == 'spherical':
            n_cov_params = n_states
        else:
            n_cov_params = n_states * n_features  # tied
        
        return n_trans_params + n_start_params + n_mean_params + n_cov_params

# Create tuner instance
tuner = HMMParameterTuner(train_data, val_data, df_signals)
print("✅ Interactive parameter tuner created")

✅ Interactive parameter tuner created


## Launch Interactive Interface

In [44]:
# Display the interactive tuning interface
interface = tuner.create_tuning_interface()
display(interface)

print("🎛️  Interactive HMM Parameter Tuning Interface")
print("=" * 50)
print("Instructions:")
print("1. Adjust parameters using the sliders and dropdowns")
print("2. Click '🚀 Train Model' to train with current parameters")
print("3. Try different configurations and compare results")
print("4. Click '📊 Compare All' to see comparison of all trained models")
print("5. Use '🔄 Reset' to clear all results and start fresh")
print("\nTip: Start with default parameters, then experiment with different values!")

🎛️  Interactive HMM Parameter Tuning Interface
Instructions:
1. Adjust parameters using the sliders and dropdowns
2. Click '🚀 Train Model' to train with current parameters
3. Try different configurations and compare results
4. Click '📊 Compare All' to see comparison of all trained models
5. Use '🔄 Reset' to clear all results and start fresh

Tip: Start with default parameters, then experiment with different values!


## Summary and Recommendations

In [45]:
print("📋 Parameter Optimization Summary")
print("=" * 60)
print("✅ Interactive parameter tuning interface created")
print("✅ Real-time model training and evaluation")
print("✅ Model comparison and visualization tools")
print("✅ Progress tracking and error handling")

print("\n🚀 Next Steps:")
print("   1. Use the interactive interface to experiment with different parameters")
print("   2. Train multiple models and compare their performance")
print("   3. Use the best configuration for regime analysis")
print("   4. Export results for production deployment")

print("\n💡 Tips for Optimization:")
print("   - Start with 2-3 states and increase gradually")
print("   - Try different covariance types (full, diag, spherical)")
print("   - Monitor convergence and training time")
print("   - Use BIC for model selection (lower is better)")
print("   - Compare multiple random seeds for robustness")

📋 Parameter Optimization Summary
✅ Interactive parameter tuning interface created
✅ Real-time model training and evaluation
✅ Model comparison and visualization tools
✅ Progress tracking and error handling

🚀 Next Steps:
   1. Use the interactive interface to experiment with different parameters
   2. Train multiple models and compare their performance
   3. Use the best configuration for regime analysis
   4. Export results for production deployment

💡 Tips for Optimization:
   - Start with 2-3 states and increase gradually
   - Try different covariance types (full, diag, spherical)
   - Monitor convergence and training time
   - Use BIC for model selection (lower is better)
   - Compare multiple random seeds for robustness
